# Week 1 — Adult Census Income Preprocessing

A reproducible, decision-oriented walkthrough for the AI Pioneers Machine Learning Internship.

## Internship Objectives

1. Understand the fundamentals of Machine Learning using Python.
2. Learn data loading, cleaning, handling missing values, feature selection, encoding categorical variables, normalization, and exploratory data analysis using Pandas and NumPy.
3. Deliverable: Clean and preprocess a sample dataset and document each preprocessing step.

## Dataset and source

This notebook uses the UCI Adult Census Income dataset (1994 U.S. Census Bureau data): https://archive.ics.uci.edu/dataset/2/adult. It combines numerical and nominal demographic/work variables and contains explicit `?` missing-value markers. The notebook calls the modular implementation in `src/`; it does not duplicate the pipeline.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_loading import load_raw_data
from src.data_cleaning import clean_data
from src.feature_selection import select_features
from src.preprocessing import preprocess_features, NUMERICAL_FEATURES, CATEGORICAL_FEATURES
from src.pipeline import run_pipeline


## Loading and initial inspection

The loader converts the source marker `?` to missing values while leaving files in `data/raw/` untouched.

In [2]:
raw = load_raw_data(ROOT / 'data/raw')
print(f'Shape: {raw.shape}')
display(raw.head())
display(raw.dtypes.rename('dtype').to_frame())
display(raw.describe(include='all').T[['count', 'unique', 'mean', 'min', 'max']])

Shape: (48842, 15)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


,dtype
age,int64
workclass,object
fnlwgt,int64
education,object
education_num,int64
marital_status,object
occupation,object
relationship,object
race,object
sex,object


,count,unique,mean,min,max
age,48842.0,NaN,38.643585,17.0,90.0
workclass,46043,8,NaN,NaN,NaN
fnlwgt,48842.0,NaN,189664.134597,12285.0,1490400.0
education,48842,16,NaN,NaN,NaN
education_num,48842.0,NaN,10.078089,1.0,16.0
marital_status,48842,7,NaN,NaN,NaN
occupation,46033,14,NaN,NaN,NaN
relationship,48842,6,NaN,NaN,NaN
race,48842,5,NaN,NaN,NaN
sex,48842,2,NaN,NaN,NaN


## Data-quality assessment and EDA

The central issues are exact duplicates, a survey-weight field, and missing categorical values. The charts generated by the pipeline answer specific questions: where data are missing, how numeric variables differ in scale/distribution, which occupations dominate, and whether numeric features are strongly redundant.

In [3]:
quality = pd.DataFrame({'missing_count': raw.isna().sum(), 'missing_percent': raw.isna().mean() * 100})
display(quality.query('missing_count > 0').sort_values('missing_count', ascending=False).round(2))
print('Exact duplicate rows:', raw.duplicated().sum())
display(raw[NUMERICAL_FEATURES + ['education_num']].corr().round(2))

,missing_count,missing_percent
occupation,2809,5.75
workclass,2799,5.73
native_country,857,1.75


Exact duplicate rows: 52


,age,capital_gain,capital_loss,hours_per_week,education_num
age,1.00,0.08,0.06,0.07,0.03
capital_gain,0.08,1.00,-0.03,0.08,0.13
capital_loss,0.06,-0.03,1.00,0.05,0.08
hours_per_week,0.07,0.08,0.05,1.00,0.14
education_num,0.03,0.13,0.08,0.14,1.00


## Preprocessing decisions

- Remove only exact duplicates.
- Remove `fnlwgt`: it is a census sampling weight, not a person-level predictor.
- Retain valid extreme capital gain/loss values; they are not data errors.
- Remove `education_num` because it duplicates `education`; remove `native_country` because it is partly missing and highly imbalanced.
- Impute retained categorical variables with the most frequent category, then one-hot encode them.
- Standardize numeric predictors; do not scale indicator columns or the income label.

In [4]:
cleaned, cleaning_log = clean_data(raw)
selected, selection_log = select_features(cleaned)
features, transformer = preprocess_features(selected)
processed = pd.concat([features.reset_index(drop=True), selected[['income']].reset_index(drop=True)], axis=1)
print('Cleaning:', cleaning_log)
print('Feature-selection reasons:', selection_log)
print('Final shape:', processed.shape)
print('Remaining missing cells:', processed.isna().sum().sum())
display(processed.head())

Cleaning: {'duplicates_removed': 52, 'columns_removed': ['fnlwgt']}
Feature-selection reasons: {'education_num': 'Exact numeric duplicate of education level; retain the readable categorical education field.', 'native_country': 'Missing in 1.8% of records and highly imbalanced (United-States dominates); not retained for a compact baseline feature set.'}
Final shape: (48790, 63)
Remaining missing cells: 0


,age,capital_gain,capital_loss,hours_per_week,workclass_Federal-gov,workclass_Local-gov,workclass_Never-worked,workclass_Private,workclass_Self-emp-inc,workclass_Self-emp-not-inc,...,relationship_Unmarried,relationship_Wife,race_Amer-Indian-Eskimo,race_Asian-Pac-Islander,race_Black,race_Other,race_White,sex_Female,sex_Male,income
0,0.025328,0.146702,-0.217248,-0.034366,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,<=50K
1,0.827758,-0.144882,-0.217248,-2.213085,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,<=50K
2,-0.047620,-0.144882,-0.217248,-0.034366,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,<=50K
3,1.046603,-0.144882,-0.217248,-0.034366,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,<=50K
4,-0.777103,-0.144882,-0.217248,-0.034366,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,<=50K


## Before/after evidence

Executing `run_pipeline()` regenerates the processed data, figures, and CSV audit tables. The table below documents the raw-to-processed transformation.

In [5]:
metadata = run_pipeline(ROOT)
display(pd.read_csv(ROOT / 'outputs/tables/pipeline_before_after.csv'))
display(pd.read_csv(ROOT / 'outputs/tables/scaling_before_after.csv').round(3))
metadata

,stage,rows,columns,missing_cells,description
0,Raw combined UCI files,48842,15,6465,Original data; '?' parsed as missing
1,After exact-duplicate removal,48790,14,6456,Removed exact repeated rows; removed fnlwgt
2,After feature selection,48790,12,5600,Removed education_num and native_country
3,Processed output,48790,63,0,"Median/mode imputation, one-hot encoding, stan..."


,Unnamed: 0,min_before,max_before,mean_before,std_before,min_after,max_after,mean_after,std_after
0,age,17.0,90.0,38.653,13.708,-1.580,3.746,-0.0,1.0
1,capital_gain,0.0,99999.0,1080.218,7455.906,-0.145,13.267,-0.0,1.0
2,capital_loss,0.0,4356.0,87.596,403.209,-0.217,10.586,0.0,1.0
3,hours_per_week,1.0,99.0,40.426,12.393,-3.181,4.727,0.0,1.0


{'raw_shape': [48842, 15],
 'cleaned_shape': [48790, 14],
 'selected_shape': [48790, 12],
 'processed_shape': [48790, 63],
 'raw_missing_cells': 6465,
 'processed_missing_cells': 0,
 'missing_by_column': {'workclass': 2799,
  'occupation': 2809,
  'native_country': 857},
 'cleaning': {'duplicates_removed': 52, 'columns_removed': ['fnlwgt']},
 'feature_selection': {'education_num': 'Exact numeric duplicate of education level; retain the readable categorical education field.',
  'native_country': 'Missing in 1.8% of records and highly imbalanced (United-States dominates); not retained for a compact baseline feature set.'},
 'numerical_features': ['age',
  'capital_gain',
  'capital_loss',
  'hours_per_week'],
 'categorical_features': ['workclass',
  'education',
  'marital_status',
  'occupation',
  'relationship',
  'race',
  'sex'],
 'encoded_feature_count': 62}

## Key findings and conclusion

The executed pipeline retains 48,790 unique records and produces a complete processed file with standardized numerical features, one-hot-encoded nominal features, and the untouched income label. This demonstrates the Week 1 workflow from inspection through documented preprocessing, while preserving raw data separately and avoiding model-training scope creep.